In [30]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import seaborn as sns

In [31]:
pd.set_option("display.max_columns", None)

In [32]:
df = sns.load_dataset('titanic')

In [33]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (891, 15)
Columns: ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [34]:
# Drop rows with missing target
df.isnull().sum()

,0
survived,0
pclass,0
sex,0
age,177
sibsp,0
parch,0
fare,0
embarked,2
class,0
who,0


In [35]:
target = "survived"

missing_target_count = df[target].isna().sum()
print("Rows with missing target:",missing_target_count)

Rows with missing target: 0


In [36]:
# drop the rows has the missing target
df = df.dropna(subset = [target])
print("after dropping:",missing_target_count)


after dropping: 0


In [37]:
# seperate X (FEATURES) and Y (TARGET)
X = df.drop(columns=[target])
y = df[target]
print(X.shape)
print(df.shape)
print(y.shape)

(891, 14)
(891, 15)
(891,)


In [38]:
print("Target distribution on full data:")
print(y.value_counts())

Target distribution on full data:
survived
0    549
1    342
Name: count, dtype: int64


In [39]:
# Train test split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)

In [40]:
print("Shape X_TRAIN:", X_train.shape)
print("Shape X_TEST:", X_test.shape)
print("Shape y_TRAIN:", y_train.shape)
print("Shape y_TEST:", y_test.shape)

Shape X_TRAIN: (712, 14)
Shape X_TEST: (179, 14)
Shape y_TRAIN: (712,)
Shape y_TEST: (179,)


In [41]:
# training set - features and target variables
X_test.head()

,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
565,3,male,24.0,2,0,24.1500,S,Third,man,True,NaN,Southampton,no,False
160,3,male,44.0,0,1,16.1000,S,Third,man,True,NaN,Southampton,no,False
553,3,male,22.0,0,0,7.2250,C,Third,man,True,NaN,Cherbourg,yes,True
860,3,male,41.0,2,0,14.1083,S,Third,man,True,NaN,Southampton,no,False
241,3,female,NaN,1,0,15.5000,Q,Third,woman,False,NaN,Queenstown,yes,False


In [42]:
y_test.head()

,survived
565,0
160,0
553,1
860,0
241,1


In [43]:
# check missing values before processing
print("Missing values in TRAIN (BEFORE):")
print(X_train.isna().mean() * 100)
print("-"*50)
print("Missing values in TEST (BEFORE):")
print(X_test.isna().mean() * 100)


Missing values in TRAIN (BEFORE):
pclass          0.000000
sex             0.000000
age            19.241573
sibsp           0.000000
parch           0.000000
fare            0.000000
embarked        0.280899
class           0.000000
who             0.000000
adult_male      0.000000
deck           77.668539
embark_town     0.280899
alive           0.000000
alone           0.000000
dtype: float64
--------------------------------------------------
Missing values in TEST (BEFORE):
pclass          0.000000
sex             0.000000
age            22.346369
sibsp           0.000000
parch           0.000000
fare            0.000000
embarked        0.000000
class           0.000000
who             0.000000
adult_male      0.000000
deck           75.418994
embark_town     0.000000
alive           0.000000
alone           0.000000
dtype: float64


In [44]:
# columns with highest missing percentage:
missing_threshold = 40
missing_percent_train = X_train.isna().mean() * 100
print("Percentage in each column: ")
print(missing_percent_train)

high_missing_cols = missing_percent_train[missing_percent_train > missing_threshold].index.tolist()
print("-"*50)
print(high_missing_cols)

Percentage in each column: 
pclass          0.000000
sex             0.000000
age            19.241573
sibsp           0.000000
parch           0.000000
fare            0.000000
embarked        0.280899
class           0.000000
who             0.000000
adult_male      0.000000
deck           77.668539
embark_town     0.280899
alive           0.000000
alone           0.000000
dtype: float64
--------------------------------------------------
['deck']


In [47]:
# X_train = X_train.drop(columns=high_missing_cols)
# X_test = X_test.drop(columns = high_missing_cols)

Index(['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class',
       'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone'],
      dtype='object')

In [56]:
# indetify numerical and categorical columns
cat_columns = X_train.select_dtypes(exclude=[np.number]).columns.tolist()
num_coloumns = X_train.select_dtypes(include = [np.number]).columns.tolist()

print("Categorical:", cat_columns)
print("Numerical:", num_coloumns)

Categorical: ['sex', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']
Numerical: ['pclass', 'age', 'sibsp', 'parch', 'fare']


In [62]:
X_train_imputed = X_train.copy()
X_test_imputed = X_test.copy()

In [64]:
numeric_median = {}
for col in num_coloumns:
  median_val = X_train_imputed[col].median()
  numeric_median[col] = median_val
  print(f"Filling numerica column {col} with train median: {median_val}")
  X_train_imputed[col] = X_train_imputed[col].fillna(median_val)
  X_test_imputed[col] = X_test_imputed[col].fillna(median_val)


print(numeric_median)

Filling numerica column pclass with train median: 3.0
Filling numerica column age with train median: 28.5
Filling numerica column sibsp with train median: 0.0
Filling numerica column parch with train median: 0.0
Filling numerica column fare with train median: 14.4542
{'pclass': 3.0, 'age': 28.5, 'sibsp': 0.0, 'parch': 0.0, 'fare': 14.4542}


In [66]:
# check missing values
print("Missing values in TRAIN (BEFORE):")
print(X_train_imputed.isna().mean() * 100)
print("-"*50)
print("Missing values in TEST (BEFORE):")
print(X_test_imputed.isna().mean() * 100)

Missing values in TRAIN (BEFORE):
pclass          0.000000
sex             0.000000
age             0.000000
sibsp           0.000000
parch           0.000000
fare            0.000000
embarked        0.280899
class           0.000000
who             0.000000
adult_male      0.000000
deck           77.668539
embark_town     0.280899
alive           0.000000
alone           0.000000
dtype: float64
--------------------------------------------------
Missing values in TEST (BEFORE):
pclass          0.000000
sex             0.000000
age             0.000000
sibsp           0.000000
parch           0.000000
fare            0.000000
embarked        0.000000
class           0.000000
who             0.000000
adult_male      0.000000
deck           75.418994
embark_town     0.000000
alive           0.000000
alone           0.000000
dtype: float64


In [70]:
cat_modes = {}

for col in cat_columns:
  cat_mode = X_test_imputed[col].mode()[0]
  print(f"Fill the NA of {col}: {cat_mode}")
  cat_modes[col] = cat_mode
  X_train_imputed[col] = X_train_imputed[col].fillna(cat_mode)
  X_test_imputed[col] = X_test_imputed[col].fillna(cat_mode)

print(cat_modes)

Fill the NA of sex: male
Fill the NA of embarked: S
Fill the NA of class: Third
Fill the NA of who: man
Fill the NA of adult_male: True
Fill the NA of deck: C
Fill the NA of embark_town: Southampton
Fill the NA of alive: no
Fill the NA of alone: True
{'sex': 'male', 'embarked': 'S', 'class': 'Third', 'who': 'man', 'adult_male': np.True_, 'deck': 'C', 'embark_town': 'Southampton', 'alive': 'no', 'alone': np.True_}


In [71]:
print("Missing values in TRAIN (BEFORE):")
print(X_train_imputed.isna().mean() * 100)
print("-"*50)
print("Missing values in TEST (BEFORE):")
print(X_test_imputed.isna().mean() * 100)

Missing values in TRAIN (BEFORE):
pclass         0.0
sex            0.0
age            0.0
sibsp          0.0
parch          0.0
fare           0.0
embarked       0.0
class          0.0
who            0.0
adult_male     0.0
deck           0.0
embark_town    0.0
alive          0.0
alone          0.0
dtype: float64
--------------------------------------------------
Missing values in TEST (BEFORE):
pclass         0.0
sex            0.0
age            0.0
sibsp          0.0
parch          0.0
fare           0.0
embarked       0.0
class          0.0
who            0.0
adult_male     0.0
deck           0.0
embark_town    0.0
alive          0.0
alone          0.0
dtype: float64


In [77]:
train_imputed_df = X_train_imputed.copy()
test_imputed_df = X_test_imputed.copy()

train_imputed_df[target] = y_train.values
test_imputed_df[target] = y_test.values
train_imputed_df

,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,survived
692,3,male,28.5,0,0,56.4958,S,Third,man,True,C,Southampton,yes,True,1
481,2,male,28.5,0,0,0.0000,S,Second,man,True,C,Southampton,no,True,0
527,1,male,28.5,0,0,221.7792,S,First,man,True,C,Southampton,no,True,0
855,3,female,18.0,0,1,9.3500,S,Third,woman,False,C,Southampton,yes,False,1
801,2,female,31.0,1,1,26.2500,S,Second,woman,False,C,Southampton,yes,False,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
359,3,female,28.5,0,0,7.8792,Q,Third,woman,False,C,Queenstown,yes,True,1
258,1,female,35.0,0,0,512.3292,C,First,woman,False,C,Cherbourg,yes,True,1
736,3,female,48.0,1,3,34.3750,S,Third,woman,False,C,Southampton,no,False,0
462,1,male,47.0,0,0,38.5000,S,First,man,True,E,Southampton,no,True,0


In [76]:
X_test_imputed

,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
565,3,male,24.0,2,0,24.1500,S,Third,man,True,C,Southampton,no,False
160,3,male,44.0,0,1,16.1000,S,Third,man,True,C,Southampton,no,False
553,3,male,22.0,0,0,7.2250,C,Third,man,True,C,Cherbourg,yes,True
860,3,male,41.0,2,0,14.1083,S,Third,man,True,C,Southampton,no,False
241,3,female,28.5,1,0,15.5000,Q,Third,woman,False,C,Queenstown,yes,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
880,2,female,25.0,0,1,26.0000,S,Second,woman,False,C,Southampton,yes,False
91,3,male,20.0,0,0,7.8542,S,Third,man,True,C,Southampton,no,True
883,2,male,28.0,0,0,10.5000,S,Second,man,True,C,Southampton,no,True
473,2,female,23.0,0,0,13.7917,C,Second,woman,False,D,Cherbourg,yes,True
